# XGBoost Ranker With Temporal Folds

This notebook implements the next training iteration for the country-ranking task:

1. train a ranking model with `XGBRanker`,
2. tune it with rolling temporal folds on the training split,
3. select hyperparameters with ranking metrics, especially `ndcg@5`,
4. evaluate the tuned model on the untouched validation split,
5. retrain on `train + val`,
6. and score the untouched test split once.

The notebook also keeps a naive popularity baseline for comparison and writes artifacts under a separate `xgboost_ranker_temporal_tuned` namespace.


In [ ]:
from pathlib import Path
import json
import os
import tempfile
import warnings

os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp(prefix="matplotlib-"))

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Install xgboost first, e.g. `pip install -r requirements.txt`.") from exc

try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    optuna = None
    OPTUNA_AVAILABLE = False

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "datasets" / "v3_features"
MODEL_DIR = ROOT / "artifacts" / "models" / "xgboost_ranker_temporal_tuned"
EVAL_DIR = ROOT / "artifacts" / "evaluations" / "xgboost_ranker_temporal_tuned"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.parquet"
VAL_PATH = DATA_DIR / "val.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

assert TRAIN_PATH.exists(), f"Missing training split: {TRAIN_PATH}"
assert VAL_PATH.exists(), f"Missing validation split: {VAL_PATH}"
assert TEST_PATH.exists(), f"Missing test split: {TEST_PATH}"

RANDOM_STATE = 42
TOP_K = 5
RUN_FULL_SPLITS = True
DEBUG_MAX_TRAIN_TRACKS = None
DEBUG_MAX_VAL_TRACKS = None
DEBUG_MAX_TEST_TRACKS = None

RUN_TUNING = True
PREFER_OPTUNA = True
N_TUNING_TRIALS = 20
N_TIME_FOLDS = 3
TIME_BLOCKS = 5
EARLY_STOPPING_ROUNDS = 50
TUNING_N_ESTIMATORS = 2000
FINAL_N_ESTIMATOR_BUFFER = 1.10

FEATURE_EXCLUDE = [
    "track_id",
    "observation_time",
    "target_country",
    "did_enter_within_60d",
    "days_to_entry",
]

TRAIN_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TRAIN_TRACKS
VAL_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_VAL_TRACKS
TEST_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TEST_TRACKS
RUN_MODE = "full" if RUN_FULL_SPLITS else "debug_sampled"

print({
    "run_mode": RUN_MODE,
    "run_tuning": RUN_TUNING,
    "n_tuning_trials": N_TUNING_TRIALS,
    "n_time_folds": N_TIME_FOLDS,
    "time_blocks": TIME_BLOCKS,
    "prefer_optuna": PREFER_OPTUNA,
    "optuna_available": OPTUNA_AVAILABLE,
    "top_k": TOP_K,
})

In [2]:
con = duckdb.connect()


def load_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f'''
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        '''
    df = con.execute(query).fetchdf()
    df["observation_time"] = pd.to_datetime(df["observation_time"])
    return df


def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    return df[feature_cols].copy().fillna(fill_values)


def prepare_ranker_inputs(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series):
    ordered = df.sort_values(["track_id", "target_country"]).reset_index(drop=True)
    X = make_feature_matrix(ordered, feature_cols, fill_values)
    y = ordered["did_enter_within_60d"].astype(float).to_numpy()
    group = ordered.groupby("track_id", sort=False).size().to_numpy()
    return ordered, X, y, group


def compute_candidate_stats(scored_df: pd.DataFrame) -> dict:
    per_track = scored_df.groupby("track_id").agg(
        candidates=("target_country", "size"),
        positives=("did_enter_within_60d", "sum"),
    )
    positive_mask = per_track["positives"] > 0
    return {
        "tracks": int(per_track.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        "avg_candidates_per_track": float(per_track["candidates"].mean()),
        "avg_future_countries_per_track": float(per_track["positives"].mean()),
        "avg_future_countries_per_positive_track": float(per_track.loc[positive_mask, "positives"].mean()) if positive_mask.any() else None,
    }


def ranking_metrics(scored_df: pd.DataFrame, k: int = 5):
    rows = []
    for track_id, group in scored_df.groupby("track_id", sort=False):
        group = group.sort_values(["score", "tie_break"], ascending=[False, False]).reset_index(drop=True)
        labels = group["did_enter_within_60d"].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top["did_enter_within_60d"].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][: len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append(
            {
                "track_id": track_id,
                "positives": positives,
                "top_k_hits": hits,
                f"precision@{k}": precision,
                f"recall@{k}": recall,
                f"hit_rate@{k}": hit_rate,
                f"ndcg@{k}": ndcg,
                f"map@{k}": map_k,
            }
        )

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df["positives"] > 0
    summary = {
        "tracks": int(metric_df.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        f"precision@{k}": float(metric_df[f"precision@{k}"].mean()),
        f"recall@{k}": float(metric_df.loc[positive_mask, f"recall@{k}"].mean()) if positive_mask.any() else None,
        f"hit_rate@{k}": float(metric_df.loc[positive_mask, f"hit_rate@{k}"].mean()) if positive_mask.any() else None,
        f"ndcg@{k}": float(metric_df.loc[positive_mask, f"ndcg@{k}"].mean()) if positive_mask.any() else None,
        f"map@{k}": float(metric_df.loc[positive_mask, f"map@{k}"].mean()) if positive_mask.any() else None,
        "mean_future_countries_per_track": float(metric_df["positives"].mean()),
    }
    return summary, metric_df


def evaluate_scores(scored_df: pd.DataFrame, k: int = 5):
    candidate_stats = compute_candidate_stats(scored_df)
    binary = {
        "roc_auc": float(roc_auc_score(scored_df["did_enter_within_60d"], scored_df["score"])),
        "average_precision": float(average_precision_score(scored_df["did_enter_within_60d"], scored_df["score"])),
    }
    ranking_all, track_metrics = ranking_metrics(scored_df, k=k)
    positive_track_metrics = track_metrics[track_metrics["positives"] > 0].copy()
    positive_summary = {
        "tracks": int(positive_track_metrics.shape[0]),
        "avg_future_countries_per_positive_track": float(positive_track_metrics["positives"].mean()) if not positive_track_metrics.empty else None,
        f"avg_top_{k}_hits_per_positive_track": float(positive_track_metrics["top_k_hits"].mean()) if not positive_track_metrics.empty else None,
        f"recall@{k}": float(positive_track_metrics[f"recall@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"hit_rate@{k}": float(positive_track_metrics[f"hit_rate@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"ndcg@{k}": float(positive_track_metrics[f"ndcg@{k}"].mean()) if not positive_track_metrics.empty else None,
        f"map@{k}": float(positive_track_metrics[f"map@{k}"].mean()) if not positive_track_metrics.empty else None,
    }
    return {
        "binary": binary,
        "candidate_stats": candidate_stats,
        "ranking_all_tracks": ranking_all,
        "ranking_positive_tracks": positive_summary,
    }, track_metrics


def normalize_scores(values: pd.Series) -> pd.Series:
    value_min = float(values.min())
    value_max = float(values.max())
    if value_max > value_min:
        return (values - value_min) / (value_max - value_min)
    return pd.Series(np.full(len(values), 0.5), index=values.index)


def score_ranker(model: xgb.XGBRanker, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, model_name: str) -> pd.DataFrame:
    ordered, X, _, _ = prepare_ranker_inputs(df, feature_cols, fill_values)
    raw_scores = pd.Series(model.predict(X), index=ordered.index)
    scored = ordered[[
        "track_id",
        "observation_time",
        "target_country",
        "did_enter_within_60d",
        "days_to_entry",
        "target_avg_daily_streams",
        "target_new_entry_rate_30d",
    ]].copy()
    scored["score"] = normalize_scores(raw_scores)
    scored["raw_score"] = raw_scores.to_numpy()
    scored["tie_break"] = scored["target_new_entry_rate_30d"].fillna(0.0)
    scored["model_name"] = model_name
    return scored


def score_naive_popularity(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[[
        "track_id",
        "observation_time",
        "target_country",
        "did_enter_within_60d",
        "days_to_entry",
        "target_avg_daily_streams",
        "target_new_entry_rate_30d",
    ]].copy()
    primary = scored["target_avg_daily_streams"].fillna(0.0)
    tie_break = scored["target_new_entry_rate_30d"].fillna(0.0)
    raw_score = primary + tie_break * 1e-6
    scored["score"] = normalize_scores(raw_score)
    scored["raw_score"] = raw_score
    scored["tie_break"] = tie_break
    scored["model_name"] = "naive_popularity_baseline"
    return scored


def feature_category(name: str) -> str:
    if name.startswith("rank_") or name == "track_in_viral50_at_obs":
        return "current_footprint"
    if name.startswith("artist_") or name == "multi_artist_flag":
        return "artist_history"
    if name.startswith("target_"):
        return "target_country_priors"
    if name in {"cultural_dist_min", "cultural_dist_missing", "same_language_flag", "same_continent_flag", "neighbor_entered_count"}:
        return "origin_target_relationship"
    if name.startswith("af_") or name in {"duration_ms", "explicit", "days_since_release", "is_friday_release"}:
        return "audio_track_metadata"
    if name.startswith("observation_"):
        return "temporal"
    return "other"


def make_temporal_folds(df: pd.DataFrame, time_blocks: int = 5):
    track_meta = (
        df[["track_id", "observation_time"]]
        .drop_duplicates()
        .sort_values(["observation_time", "track_id"])
        .reset_index(drop=True)
    )
    block_indices = [idx for idx in np.array_split(np.arange(len(track_meta)), time_blocks) if len(idx) > 0]
    blocks = [track_meta.iloc[idx].copy().reset_index(drop=True) for idx in block_indices]
    if len(blocks) < 4:
        raise ValueError(f"Need at least 4 non-empty time blocks, found {len(blocks)}")

    folds = []
    for fold_idx in range(len(blocks) - 2):
        train_block_df = pd.concat(blocks[: fold_idx + 2], ignore_index=True)
        val_block_df = blocks[fold_idx + 2].reset_index(drop=True)
        folds.append(
            {
                "fold": fold_idx + 1,
                "train_track_ids": train_block_df["track_id"].tolist(),
                "val_track_ids": val_block_df["track_id"].tolist(),
                "train_tracks": int(train_block_df.shape[0]),
                "val_tracks": int(val_block_df.shape[0]),
                "train_start": train_block_df["observation_time"].min(),
                "train_end": train_block_df["observation_time"].max(),
                "val_start": val_block_df["observation_time"].min(),
                "val_end": val_block_df["observation_time"].max(),
            }
        )
    return folds[-N_TIME_FOLDS:]


def make_ranker(params: dict, n_estimators: int = TUNING_N_ESTIMATORS, early_stopping_rounds: int | None = EARLY_STOPPING_ROUNDS):
    init_kwargs = {
        "objective": "rank:ndcg",
        "eval_metric": f"ndcg@{TOP_K}",
        "tree_method": "hist",
        "n_estimators": n_estimators,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        **params,
    }
    if early_stopping_rounds is not None:
        init_kwargs["early_stopping_rounds"] = early_stopping_rounds
    return xgb.XGBRanker(**init_kwargs)


def sample_random_params(rng: np.random.Generator) -> dict:
    return {
        "learning_rate": float(np.exp(rng.uniform(np.log(0.02), np.log(0.1)))),
        "max_depth": int(rng.integers(4, 11)),
        "min_child_weight": float(np.exp(rng.uniform(np.log(1.0), np.log(50.0)))),
        "subsample": float(rng.uniform(0.6, 1.0)),
        "colsample_bytree": float(rng.uniform(0.5, 1.0)),
        "gamma": float(rng.uniform(0.0, 5.0)),
        "reg_alpha": float(np.exp(rng.uniform(np.log(1e-4), np.log(10.0)))),
        "reg_lambda": float(np.exp(rng.uniform(np.log(0.1), np.log(20.0)))),
    }


In [3]:
train_df = load_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
val_df = load_split(VAL_PATH, VAL_MAX_TRACKS)
test_df = load_split(TEST_PATH, TEST_MAX_TRACKS)

feature_cols = [col for col in train_df.columns if col not in FEATURE_EXCLUDE]
fill_values_train = train_df[feature_cols].median(numeric_only=True)

time_folds = make_temporal_folds(train_df, time_blocks=TIME_BLOCKS)
fold_df = pd.DataFrame(
    [
        {
            "fold": fold["fold"],
            "train_tracks": fold["train_tracks"],
            "val_tracks": fold["val_tracks"],
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "val_start": fold["val_start"],
            "val_end": fold["val_end"],
        }
        for fold in time_folds
    ]
)

print(f"Train rows: {len(train_df):,} | tracks: {train_df['track_id'].nunique():,}")
print(f"Val rows:   {len(val_df):,} | tracks: {val_df['track_id'].nunique():,}")
print(f"Test rows:  {len(test_df):,} | tracks: {test_df['track_id'].nunique():,}")
print(f"Feature count: {len(feature_cols)}")
print("Temporal CV folds:")
display(fold_df)


Train rows: 272,922 | tracks: 64,093
Val rows:   1,647,221 | tracks: 25,858
Test rows:  1,535,867 | tracks: 24,047
Feature count: 103
Temporal CV folds:


,fold,train_tracks,val_tracks,train_start,train_end,val_start,val_end
0,1,25638,12819,2017-01-01,2018-04-06,2018-04-06,2018-11-16
1,2,38457,12818,2017-01-01,2018-11-16,2018-11-16,2019-06-21
2,3,51275,12818,2017-01-01,2019-06-21,2019-06-21,2019-12-31


In [4]:
default_ranker_params = {
    "learning_rate": 0.05,
    "max_depth": 8,
    "min_child_weight": 10.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}

trial_records = []
fold_records = []


def evaluate_param_set(params: dict, trial_label: str) -> dict:
    fold_summaries = []
    for fold in time_folds:
        fold_train = train_df[train_df["track_id"].isin(fold["train_track_ids"])].copy()
        fold_val = train_df[train_df["track_id"].isin(fold["val_track_ids"])].copy()

        fold_fill_values = fold_train[feature_cols].median(numeric_only=True)
        _, X_train_fold, y_train_fold, group_train_fold = prepare_ranker_inputs(fold_train, feature_cols, fold_fill_values)
        _, X_val_fold, y_val_fold, group_val_fold = prepare_ranker_inputs(fold_val, feature_cols, fold_fill_values)

        model = make_ranker(params)
        model.fit(
            X_train_fold,
            y_train_fold,
            group=group_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_group=[group_val_fold],
            verbose=False,
        )

        scored_val = score_ranker(model, fold_val, feature_cols, fold_fill_values, model_name="xgb_ranker_fold")
        summary, _ = evaluate_scores(scored_val, k=TOP_K)
        best_iteration = getattr(model, "best_iteration", None)
        resolved_rounds = int(best_iteration + 1) if best_iteration is not None else TUNING_N_ESTIMATORS

        fold_record = {
            "trial_label": trial_label,
            "fold": fold["fold"],
            "best_iteration": resolved_rounds,
            "roc_auc": summary["binary"]["roc_auc"],
            "average_precision": summary["binary"]["average_precision"],
            f"precision@{TOP_K}": summary["ranking_all_tracks"][f"precision@{TOP_K}"],
            f"recall@{TOP_K}": summary["ranking_all_tracks"][f"recall@{TOP_K}"],
            f"hit_rate@{TOP_K}": summary["ranking_all_tracks"][f"hit_rate@{TOP_K}"],
            f"ndcg@{TOP_K}": summary["ranking_all_tracks"][f"ndcg@{TOP_K}"],
            f"map@{TOP_K}": summary["ranking_all_tracks"][f"map@{TOP_K}"],
        }
        fold_summaries.append(fold_record)
        fold_records.append(fold_record | params)

    fold_df_local = pd.DataFrame(fold_summaries)
    return {
        "trial_label": trial_label,
        "mean_best_iteration": float(fold_df_local["best_iteration"].mean()),
        "mean_roc_auc": float(fold_df_local["roc_auc"].mean()),
        "mean_average_precision": float(fold_df_local["average_precision"].mean()),
        f"mean_precision@{TOP_K}": float(fold_df_local[f"precision@{TOP_K}"].mean()),
        f"mean_recall@{TOP_K}": float(fold_df_local[f"recall@{TOP_K}"].mean()),
        f"mean_hit_rate@{TOP_K}": float(fold_df_local[f"hit_rate@{TOP_K}"].mean()),
        f"mean_ndcg@{TOP_K}": float(fold_df_local[f"ndcg@{TOP_K}"].mean()),
        f"mean_map@{TOP_K}": float(fold_df_local[f"map@{TOP_K}"].mean()),
        **params,
    }


if RUN_TUNING:
    if PREFER_OPTUNA and OPTUNA_AVAILABLE:
        def objective(trial):
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 4, 10),
                "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
            }
            summary = evaluate_param_set(params, trial_label=f"optuna_{trial.number:03d}")
            trial_records.append(summary)
            trial.set_user_attr(f"mean_recall@{TOP_K}", summary[f"mean_recall@{TOP_K}"])
            trial.set_user_attr(f"mean_map@{TOP_K}", summary[f"mean_map@{TOP_K}"])
            trial.set_user_attr("mean_best_iteration", summary["mean_best_iteration"])
            return summary[f"mean_ndcg@{TOP_K}"]

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=N_TUNING_TRIALS, show_progress_bar=False)
        tuning_method = "optuna"
        best_params = study.best_params
    else:
        rng = np.random.default_rng(RANDOM_STATE)
        tuning_method = "random_search"
        for trial_idx in range(N_TUNING_TRIALS):
            sampled_params = sample_random_params(rng)
            summary = evaluate_param_set(sampled_params, trial_label=f"random_{trial_idx:03d}")
            trial_records.append(summary)
        best_params = max(trial_records, key=lambda row: row[f"mean_ndcg@{TOP_K}"])
        best_params = {key: best_params[key] for key in default_ranker_params}
else:
    tuning_method = "default_params"
    best_params = default_ranker_params.copy()
    trial_records.append(evaluate_param_set(best_params, trial_label="default_params"))

trial_results_df = pd.DataFrame(trial_records).sort_values(f"mean_ndcg@{TOP_K}", ascending=False).reset_index(drop=True)
fold_results_df = pd.DataFrame(fold_records)

best_trial_row = trial_results_df.iloc[0].to_dict()
print(f"Selected tuning method: {tuning_method}")
print("Best parameter set:")
print(json.dumps(best_params, indent=2))
print()
print("Top tuning results:")
display(trial_results_df.head(10))


Selected tuning method: random_search
Best parameter set:
{
  "learning_rate": 0.07303585909985359,
  "max_depth": 9,
  "min_child_weight": 3.0893050901115746,
  "subsample": 0.8729982015899902,
  "colsample_bytree": 0.5698762418046549,
  "gamma": 0.9995410123755416,
  "reg_alpha": 0.0001088457180562647,
  "reg_lambda": 6.467502376208379
}

Top tuning results:


,trial_label,mean_best_iteration,mean_roc_auc,mean_average_precision,mean_precision@5,mean_recall@5,mean_hit_rate@5,mean_ndcg@5,mean_map@5,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,random_006,112.666667,0.848438,0.500969,0.055706,0.847145,0.997546,0.941746,0.917226,0.073036,9,3.089305,0.872998,0.569876,0.999541,0.000109,6.467502
1,random_003,115.000000,0.853241,0.502663,0.055727,0.847650,0.998012,0.941411,0.916565,0.084197,7,21.011169,0.677855,0.733361,0.219019,0.000591,3.730063
2,random_008,101.666667,0.856671,0.509058,0.055597,0.845948,0.997546,0.941225,0.916550,0.058644,8,9.126814,0.906000,0.817359,2.767897,0.062522,0.500493
3,random_004,127.000000,0.851669,0.508653,0.055732,0.847172,0.997800,0.940661,0.916002,0.066313,6,3.577400,0.748184,0.734778,0.947357,0.000446,1.243401
4,random_019,46.666667,0.852196,0.501525,0.055654,0.846523,0.997293,0.940008,0.914807,0.050194,10,6.374473,0.706790,0.665784,2.603362,0.015652,0.112132
5,random_007,175.333333,0.856846,0.516512,0.055649,0.846484,0.997546,0.939803,0.914542,0.058310,6,15.778104,0.912292,0.729458,2.843706,0.000500,0.183458
6,random_012,136.333333,0.852309,0.499526,0.055665,0.846351,0.997081,0.939636,0.914541,0.025556,7,5.728041,0.752408,0.650756,3.151413,0.006443,0.159105
7,random_001,297.666667,0.860284,0.509867,0.055592,0.845977,0.997292,0.939309,0.913887,0.024580,7,5.823609,0.748319,0.963382,3.219326,1.299595,1.047875
8,random_014,97.666667,0.854594,0.499383,0.055561,0.844692,0.996531,0.939108,0.913834,0.063404,9,2.900883,0.638556,0.951301,2.278881,0.001028,0.505843
9,random_017,256.000000,0.859939,0.507766,0.055587,0.845403,0.996785,0.938649,0.913047,0.023621,8,9.963009,0.668237,0.962560,2.905306,0.005424,2.289361


In [5]:
def fit_ranker_with_validation(train_part: pd.DataFrame, val_part: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict):
    _, X_train_part, y_train_part, group_train_part = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    _, X_val_part, y_val_part, group_val_part = prepare_ranker_inputs(val_part, feature_cols, fill_values)
    model = make_ranker(params)
    model.fit(
        X_train_part,
        y_train_part,
        group=group_train_part,
        eval_set=[(X_val_part, y_val_part)],
        eval_group=[group_val_part],
        verbose=False,
    )
    return model


def fit_final_ranker(train_part: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict, n_estimators: int):
    _, X_train_part, y_train_part, group_train_part = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    model = make_ranker(params, n_estimators=n_estimators, early_stopping_rounds=None)
    model.fit(X_train_part, y_train_part, group=group_train_part, verbose=False)
    return model


tuned_train_model = fit_ranker_with_validation(train_df, val_df, feature_cols, fill_values_train, best_params)
train_val_best_iteration = getattr(tuned_train_model, "best_iteration", None)
train_val_best_rounds = int(train_val_best_iteration + 1) if train_val_best_iteration is not None else TUNING_N_ESTIMATORS

validation_predictions = {
    "xgb_ranker_tuned": score_ranker(tuned_train_model, val_df, feature_cols, fill_values_train, model_name="xgb_ranker_tuned"),
    "naive_popularity_baseline": score_naive_popularity(val_df),
}

validation_results = {}
validation_track_tables = {}
for model_name, scored_df in validation_predictions.items():
    summary, track_metrics = evaluate_scores(scored_df, k=TOP_K)
    validation_results[model_name] = summary
    validation_track_tables[model_name] = track_metrics

combined_train_val_df = pd.concat([train_df, val_df], ignore_index=True)
fill_values_final = combined_train_val_df[feature_cols].median(numeric_only=True)
final_n_estimators = int(np.ceil(max(best_trial_row["mean_best_iteration"], train_val_best_rounds) * FINAL_N_ESTIMATOR_BUFFER))
final_model = fit_final_ranker(combined_train_val_df, feature_cols, fill_values_final, best_params, n_estimators=final_n_estimators)

test_predictions = {
    "xgb_ranker_final": score_ranker(final_model, test_df, feature_cols, fill_values_final, model_name="xgb_ranker_final"),
    "naive_popularity_baseline": score_naive_popularity(test_df),
}

test_results = {}
test_track_tables = {}
for model_name, scored_df in test_predictions.items():
    summary, track_metrics = evaluate_scores(scored_df, k=TOP_K)
    test_results[model_name] = summary
    test_track_tables[model_name] = track_metrics

comparison_rows = []
for split_name, result_map in [("validation", validation_results), ("test", test_results)]:
    for model_name, summary in result_map.items():
        comparison_rows.append({
            "split": split_name,
            "model": model_name,
            "roc_auc": summary["binary"]["roc_auc"],
            "average_precision": summary["binary"]["average_precision"],
            f"precision@{TOP_K}": summary["ranking_all_tracks"][f"precision@{TOP_K}"],
            f"recall@{TOP_K}": summary["ranking_all_tracks"][f"recall@{TOP_K}"],
            f"hit_rate@{TOP_K}": summary["ranking_all_tracks"][f"hit_rate@{TOP_K}"],
            f"ndcg@{TOP_K}": summary["ranking_all_tracks"][f"ndcg@{TOP_K}"],
            f"map@{TOP_K}": summary["ranking_all_tracks"][f"map@{TOP_K}"],
            f"positive_recall@{TOP_K}": summary["ranking_positive_tracks"][f"recall@{TOP_K}"],
            f"positive_hit_rate@{TOP_K}": summary["ranking_positive_tracks"][f"hit_rate@{TOP_K}"],
            f"positive_ndcg@{TOP_K}": summary["ranking_positive_tracks"][f"ndcg@{TOP_K}"],
            f"positive_map@{TOP_K}": summary["ranking_positive_tracks"][f"map@{TOP_K}"],
            "tracks": summary["candidate_stats"]["tracks"],
            "positive_tracks": summary["candidate_stats"]["positive_tracks"],
            "avg_candidates_per_track": summary["candidate_stats"]["avg_candidates_per_track"],
        })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["split", "model"]).reset_index(drop=True)
print("Validation/test comparison:")
display(comparison_df)

positive_rows = []
for split_name, result_map in [("validation", validation_results), ("test", test_results)]:
    for model_name, summary in result_map.items():
        positive_rows.append({
            "split": split_name,
            "model": model_name,
            "positive_tracks": summary["ranking_positive_tracks"]["tracks"],
            "avg_future_countries_per_positive_track": summary["ranking_positive_tracks"]["avg_future_countries_per_positive_track"],
            f"avg_top_{TOP_K}_hits_per_positive_track": summary["ranking_positive_tracks"][f"avg_top_{TOP_K}_hits_per_positive_track"],
            f"recall@{TOP_K}": summary["ranking_positive_tracks"][f"recall@{TOP_K}"],
            f"hit_rate@{TOP_K}": summary["ranking_positive_tracks"][f"hit_rate@{TOP_K}"],
            f"ndcg@{TOP_K}": summary["ranking_positive_tracks"][f"ndcg@{TOP_K}"],
            f"map@{TOP_K}": summary["ranking_positive_tracks"][f"map@{TOP_K}"],
        })
positive_analysis_df = pd.DataFrame(positive_rows).sort_values(["split", "model"]).reset_index(drop=True)
print()
print("Positive-track-only view:")
display(positive_analysis_df)


Validation/test comparison:


,split,model,roc_auc,average_precision,precision@5,recall@5,hit_rate@5,ndcg@5,map@5,positive_recall@5,positive_hit_rate@5,positive_ndcg@5,positive_map@5,tracks,positive_tracks,avg_candidates_per_track
0,test,naive_popularity_baseline,0.489975,0.006399,0.007036,0.071285,0.241654,0.105515,0.078229,0.071285,0.241654,0.105515,0.078229,24047,2007,63.869381
1,test,xgb_ranker_final,0.864139,0.055705,0.030831,0.669348,0.875436,0.725118,0.673948,0.669348,0.875436,0.725118,0.673948,24047,2007,63.869381
2,validation,naive_popularity_baseline,0.498401,0.006116,0.007000,0.091933,0.259030,0.112907,0.080826,0.091933,0.259030,0.112907,0.080826,25858,2104,63.702568
3,validation,xgb_ranker_tuned,0.863146,0.054736,0.028742,0.638393,0.852186,0.682894,0.625951,0.638393,0.852186,0.682894,0.625951,25858,2104,63.702568



Positive-track-only view:


,split,model,positive_tracks,avg_future_countries_per_positive_track,avg_top_5_hits_per_positive_track,recall@5,hit_rate@5,ndcg@5,map@5
0,test,naive_popularity_baseline,2007,5.165421,0.421525,0.071285,0.241654,0.105515,0.078229
1,test,xgb_ranker_final,2007,5.165421,1.847035,0.669348,0.875436,0.725118,0.673948
2,validation,naive_popularity_baseline,2104,4.951996,0.430133,0.091933,0.259030,0.112907,0.080826
3,validation,xgb_ranker_tuned,2104,4.951996,1.766160,0.638393,0.852186,0.682894,0.625951


In [6]:
booster = final_model.get_booster()
importance_gain = booster.get_score(importance_type="gain")
importance_weight = booster.get_score(importance_type="weight")

importance_df = pd.DataFrame({"feature": feature_cols})
importance_df["gain"] = importance_df["feature"].map(importance_gain).fillna(0.0)
importance_df["weight"] = importance_df["feature"].map(importance_weight).fillna(0.0)
importance_df["category"] = importance_df["feature"].map(feature_category)
importance_df = importance_df.sort_values(["gain", "weight"], ascending=[False, False]).reset_index(drop=True)

print("Top 20 final-model features by gain:")
display(importance_df.head(20))

try:
    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    plot_gain = importance_df.head(20).sort_values("gain")
    plot_weight = importance_df.sort_values("weight", ascending=False).head(20).sort_values("weight")
    axes[0].barh(plot_gain["feature"], plot_gain["gain"])
    axes[0].set_title("Top 20 Feature Importance (gain)")
    axes[0].set_xlabel("gain")
    axes[1].barh(plot_weight["feature"], plot_weight["weight"])
    axes[1].set_title("Top 20 Feature Importance (weight)")
    axes[1].set_xlabel("weight")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
except Exception as exc:
    print(f"Skipping feature importance plots after runtime error: {exc}")

category_summary = (
    importance_df.groupby("category")[["gain", "weight"]]
    .sum()
    .sort_values("gain", ascending=False)
)
print("Feature category summary:")
display(category_summary)


Top 20 final-model features by gain:


,feature,gain,weight,category
0,same_language_flag,341.882141,115.0,origin_target_relationship
1,same_continent_flag,104.996086,180.0,origin_target_relationship
2,artist_prior_success_in_target,18.776670,913.0,artist_history
3,artist_prior_unique_regions,13.943842,591.0,artist_history
4,target_continent_europe,11.910856,239.0,target_country_priors
5,target_continent_north_america,11.322805,208.0,target_country_priors
6,rank_brazil,11.068498,93.0,current_footprint
7,target_continent_south_america,11.035072,223.0,target_country_priors
8,neighbor_entered_count,9.704684,231.0,origin_target_relationship
9,rank_slovakia,9.164964,101.0,current_footprint


Feature category summary:


/var/folders/bl/m0zs7txj4lg0kmvg2zzypg840000gn/T/ipykernel_34429/189190747.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,gain,weight
category,,
origin_target_relationship,471.866988,2262.0
current_footprint,277.889449,6864.0
target_country_priors,71.733911,6171.0
artist_history,58.307221,3303.0
audio_track_metadata,31.259909,4774.0
temporal,9.489529,726.0


In [7]:
training_summary = {
    "config": {
        "run_mode": RUN_MODE,
        "run_tuning": RUN_TUNING,
        "prefer_optuna": PREFER_OPTUNA,
        "optuna_available": OPTUNA_AVAILABLE,
        "n_tuning_trials": N_TUNING_TRIALS,
        "n_time_folds": N_TIME_FOLDS,
        "time_blocks": TIME_BLOCKS,
        "top_k": TOP_K,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "tuning_n_estimators": TUNING_N_ESTIMATORS,
        "final_n_estimator_buffer": FINAL_N_ESTIMATOR_BUFFER,
        "final_n_estimators": final_n_estimators,
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "val_path": str(VAL_PATH),
        "test_path": str(TEST_PATH),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_tracks": int(train_df["track_id"].nunique()),
        "val_tracks": int(val_df["track_id"].nunique()),
        "test_tracks": int(test_df["track_id"].nunique()),
    },
    "tuning": {
        "method": tuning_method,
        "best_params": best_params,
        "best_trial": best_trial_row,
    },
    "train_only_validation_fit": {
        "best_iteration": train_val_best_rounds,
    },
    "feature_cols": feature_cols,
    "fill_values_train": {col: float(fill_values_train[col]) for col in feature_cols},
    "fill_values_final": {col: float(fill_values_final[col]) for col in feature_cols},
}

evaluation_summary = {
    "validation": validation_results,
    "test": test_results,
}

model_path = MODEL_DIR / "xgb_ranker.json"
training_summary_path = MODEL_DIR / "training_summary.json"
evaluation_summary_path = EVAL_DIR / "evaluation_summary.json"
comparison_path = EVAL_DIR / "model_vs_baseline_comparison.parquet"
positive_analysis_path = EVAL_DIR / "positive_track_analysis.parquet"
trial_results_path = EVAL_DIR / "tuning_trials.parquet"
fold_results_path = EVAL_DIR / "tuning_fold_metrics.parquet"
importance_path = EVAL_DIR / "feature_importance.parquet"
val_predictions_path = EVAL_DIR / "val_predictions.parquet"
test_predictions_path = EVAL_DIR / "test_predictions.parquet"
val_track_metrics_path = EVAL_DIR / "val_track_metrics.parquet"
test_track_metrics_path = EVAL_DIR / "test_track_metrics.parquet"
val_positive_track_metrics_path = EVAL_DIR / "val_track_metrics_positive_only.parquet"
test_positive_track_metrics_path = EVAL_DIR / "test_track_metrics_positive_only.parquet"

final_model.save_model(model_path)
with open(training_summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)
with open(evaluation_summary_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

con.register("comparison_df", comparison_df)
con.register("positive_analysis_df", positive_analysis_df)
con.register("trial_results_df", trial_results_df)
con.register("fold_results_df", fold_results_df)
con.register("importance_df", importance_df)
con.register("val_scored_df", validation_predictions["xgb_ranker_tuned"])
con.register("test_scored_df", test_predictions["xgb_ranker_final"])
con.register("val_track_metrics_df", validation_track_tables["xgb_ranker_tuned"])
con.register("test_track_metrics_df", test_track_tables["xgb_ranker_final"])
con.register(
    "val_positive_track_metrics_df",
    validation_track_tables["xgb_ranker_tuned"][validation_track_tables["xgb_ranker_tuned"]["positives"] > 0],
)
con.register(
    "test_positive_track_metrics_df",
    test_track_tables["xgb_ranker_final"][test_track_tables["xgb_ranker_final"]["positives"] > 0],
)

con.execute(f"COPY comparison_df TO '{comparison_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY positive_analysis_df TO '{positive_analysis_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY trial_results_df TO '{trial_results_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY fold_results_df TO '{fold_results_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY importance_df TO '{importance_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_scored_df TO '{val_predictions_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_scored_df TO '{test_predictions_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_track_metrics_df TO '{val_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_track_metrics_df TO '{test_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_positive_track_metrics_df TO '{val_positive_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_positive_track_metrics_df TO '{test_positive_track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")

for name in [
    "comparison_df",
    "positive_analysis_df",
    "trial_results_df",
    "fold_results_df",
    "importance_df",
    "val_scored_df",
    "test_scored_df",
    "val_track_metrics_df",
    "test_track_metrics_df",
    "val_positive_track_metrics_df",
    "test_positive_track_metrics_df",
]:
    con.unregister(name)

print(f"Saved tuned ranker to: {model_path}")
print(f"Saved training summary to: {training_summary_path}")
print(f"Saved evaluation summary to: {evaluation_summary_path}")
print(f"Saved tuning trials to: {trial_results_path}")
print(f"Saved tuning fold metrics to: {fold_results_path}")
print(f"Saved comparison table to: {comparison_path}")


Saved tuned ranker to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/models/xgboost_ranker_temporal_tuned/xgb_ranker.json
Saved training summary to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/models/xgboost_ranker_temporal_tuned/training_summary.json
Saved evaluation summary to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/evaluations/xgboost_ranker_temporal_tuned/evaluation_summary.json
Saved tuning trials to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/evaluations/xgboost_ranker_temporal_tuned/tuning_trials.parquet
Saved tuning fold metrics to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/evaluations/xgboost_ranker_temporal_tuned/tuning_fold_metrics.parquet
Saved comparison table to: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/artifacts/evalua